# Tutorial 2 — FOL Formulas: Connectives & Quantifiers

Logical connectives (¬, ∧, ∨, →, ↔) and quantifiers (∀, ∃) are evaluated as
**fuzzy operations on truth-value tensors**.

The fuzzy operators live in a `Logic` bundle attached to an `Interpretation` (`interp.logic`).
Formula strings are parsed once (outside any loop) and re-evaluated on different data.

| Pattern | String syntax |
|---|---|
| Negation | `¬IsDog(x)`, `not IsDog(x)`, `~IsDog(x)` |
| Conjunction | `IsDog(x) ∧ IsCat(x)`, `&`, `and` |
| Disjunction | `IsDog(x) ∨ IsCat(x)`, `or` |
| Implication | `IsDog(x) → IsLarge(x)`, `->`, `=>` |
| Equivalence | `IsDog(x) ↔ ¬IsCat(x)`, `<->` |
| Universal | `∀ x: IsDog(x)`, `forall x: IsDog(x)` |
| Existential | `∃ x: IsDog(x)`, `exists x: IsDog(x)` |
| Diagonal (paired) | `∀ (x, y): Similar(x, y)` |
| Guarded | `∀ x \| IsLarge(x): IsDog(x)` |

`|` is the guard separator, **not** "or": `IsDog(x) | IsCat(x)` is a parse error. Write `or` or `∨`.

In [ ]:
import torch
import sltn
from sltn import Signature, Interpretation, Logic

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Setup

The same animals as in Tutorial 1: an animal is a vector in ℝ² of two features, its **size** and its **furriness**. `IsDog` and `IsCat` are true near a dog and a cat prototype, and `Similar` compares two animals.

In [ ]:
sig = Signature("animals")
sig.sort("Animal")
sig.variable("x", "Animal")
sig.variable("y", "Animal")
sig.predicate("IsDog", ["Animal"])
sig.predicate("IsCat", ["Animal"])
sig.predicate("Similar", ["Animal", "Animal"])
print(sig)

interp = Interpretation(sig).to(device)
interp["Animal"] = sltn.Type("R2", shape=(2,))   # (size, furriness)

# Fixed predicates: the closer to the prototype, the more true
dog = torch.tensor([0.0, 0.0], device=device)
cat = torch.tensor([1.0, 1.0], device=device)
interp["IsDog"]   = lambda a: torch.exp(-torch.norm(a - dog, dim=-1, keepdim=True))
interp["IsCat"]   = lambda a: torch.exp(-torch.norm(a - cat, dim=-1, keepdim=True))
interp["Similar"] = lambda a, b: torch.exp(-torch.norm(a - b, dim=-1, keepdim=True))

# 8 animals for x, 6 for y
interp["x"] = torch.randn(8, 2, device=device)
interp["y"] = torch.randn(6, 2, device=device)

print(interp)

## 1 — Connectives

Each connective and quantifier is computed by an operator of the interpretation's `Logic`, `interp.logic`. Print it to see which operators are used:

In [ ]:
print(interp.logic)

This is the **stable product configuration**, the default of every interpretation:

- `∧` is the product `a·b`, `∨` its dual, the probabilistic sum `a + b − a·b`, and `¬` is `1 − a`.
- `→` is **Reichenbach**, `1 − a + a·b`, as in LTN and LTNtorch. It is 1 only when `a = 0` or `b = 1`, so even `a → a` is below 1 (0.75 at `a = 0.5`). Tutorial 3 compares it with the other implications; section 5 shows how to switch.
- `↔` is `(a → b) ∧ (b → a)`, with the two operators above.
- `∀` and `∃` are p-means with p = 2 (section 2). `SatAgg` combines the clauses of a knowledge base (Tutorial 4).
- **`stable=True`** keeps the inputs `eps = 1e-6` away from 0 and 1, where some operators have an infinite or NaN gradient. It changes the values by at most `eps`. Every operator is stable by default, also when you build one yourself (section 5).

`sig.parse` checks a formula and builds its tree; calling the interpretation on it, `interp(phi)`, applies the operators.

With `x` free, a formula has **one truth degree per animal** (shape `[8, 1]`). The table shows the first 4 animals: every column is computed animal by animal from the first two. For example, `∧` is the product of the `IsDog` and `IsCat` columns, and `∨` is `IsDog + IsCat − IsDog·IsCat`.

In [ ]:
formulas = {
    "IsDog":         "IsDog(x)",
    "IsCat":         "IsCat(x)",
    "¬IsDog":        "not IsDog(x)",
    "IsDog∧IsCat":   "IsDog(x) & IsCat(x)",       # a dog and a cat at once: low
    "IsDog∨IsCat":   "IsDog(x) ∨ IsCat(x)",
    "IsDog→¬IsCat":  "IsDog(x) → not IsCat(x)",   # a dog is not a cat
    "IsDog↔¬IsCat":  "IsDog(x) ↔ not IsCat(x)",
}
values = {label: interp(formula).value for label, formula in formulas.items()}
print("shape of each result:", list(values["IsDog"].shape))

print("\nanimal " + "".join(f"{label:>14s}" for label in formulas))
for i in range(4):
    print(f"{i:>6d} " + "".join(f"{v[i].item():>14.3f}" for v in values.values()))

## 2 — Quantifiers

`∀ x: φ(x)` aggregates over the axis of `x` with **pMeanError**, p = 2: `1 − (mean (1 − φ)²)^½`. It is driven by the individuals that satisfy φ least: a few bad ones lower it a lot.
`∃ x: φ(x)` uses **pMean**, p = 2: `(mean φ²)^½`, driven by the individuals that satisfy φ most.
A larger p brings them closer to min and max. The axis of `x` disappears: the result is a single truth degree.

With a binary predicate, each quantifier removes only the axis of its own variable. `Similar(x, y)` has one truth degree per pair, on axes `x` (8 animals) and `y` (6). Quantifying over one variable leaves the axis of the other, and quantifying over both gives a single truth degree:

In [ ]:
r_all = interp("forall x: IsDog(x)")    # every animal is a dog
r_ex  = interp("exists x: IsDog(x)")    # some animal is a dog
print("∀ x: IsDog(x) :", round(r_all.value.item(), 4), "  shape:", list(r_all.value.shape))
print("∃ x: IsDog(x) :", round(r_ex.value.item(),  4), "  shape:", list(r_ex.value.shape))

print()
for formula in [
    "Similar(x, y)",                      # one truth degree per pair (x, y)
    "forall x: Similar(x, y)",            # per y: is it similar to every x?
    "exists y: Similar(x, y)",            # per x: is it similar to some y?
    "forall x: exists y: Similar(x, y)",  # every x is similar to some y
]:
    r = interp(formula)
    print(f"{formula:<36s} axes {[a.name for a in r.axes]}   shape {list(r.value.shape)}")

A quantifier binds only to the smallest following expression.
`∀ x: IsDog(x) → ¬IsCat(x)` therefore means `(∀ x: IsDog(x)) → ¬IsCat(x)`, **not** "no dog is a cat", `∀ x: (IsDog(x) → ¬IsCat(x))`. To scope the quantifier over the whole implication, add parentheses. sLTN warns when a variable occurs both quantified and free, as in the first formula below.

In [ ]:
no_parens = interp("forall x: IsDog(x) -> not IsCat(x)")     # (forall x: IsDog(x)) -> not IsCat(x)
scoped    = interp("forall x: (IsDog(x) -> not IsCat(x))")   # no dog is a cat

print("forall x: IsDog(x) -> not IsCat(x)   -> shape:", list(no_parens.value.shape))   # x is still free
print("forall x: (IsDog(x) -> not IsCat(x)) -> shape:", list(scoped.value.shape))      # one truth degree

## 3 — Diagonal (paired) quantification

`∀ (x, y): φ(x, y)` pairs the variables: the $i$-th individual of `x` meets only the $i$-th individual of `y`, instead of every individual of `y` (no Cartesian product). Both variables must have the same number of individuals.

Here `y[i]` is a second photo of animal `x[i]`: the same features plus a little noise. "Every animal looks like its own photo" is the diagonal `∀ (x, y): Similar(x, y)`, and it is true. The Cartesian `∀ x: ∀ y: Similar(x, y)` compares every animal with every photo, and it is false.

In [ ]:
interp["x"] = torch.randn(8, 2, device=device)
interp["y"] = interp["x"] + 0.1 * torch.randn(8, 2, device=device)   # y[i]: a photo of x[i]

print("∀ (x, y): Similar(x, y)   [paired]   :", round(interp("∀ (x, y): Similar(x, y)").value.item(), 4))
print("∀ x: ∀ y: Similar(x, y)   [Cartesian]:", round(interp("∀ x: ∀ y: Similar(x, y)").value.item(), 4))

To see which truth degrees each one aggregates, evaluate the formula without the quantifier. `Similar(x, y)` has one truth degree per pair, on the full 8 × 8 grid. The Cartesian `∀ x: ∀ y` aggregates all 64 of them; the diagonal `∀ (x, y)` aggregates only the 8 on the diagonal, where animal `i` meets its own photo `i`. Applying the `∀` operator by hand to each set gives the same results as above.

In [ ]:
grid = interp("Similar(x, y)").value[..., 0]   # [8, 8]: one truth degree per pair (x[i], y[j])
paired = torch.diagonal(grid)                   # [8]: only the pairs (x[i], y[i])
print("Cartesian:", grid.numel(), "truth degrees,  diagonal:", paired.numel(), "truth degrees")

forall = interp.logic["∀"]
print(f"∀ over the 64 pairs  : {forall(grid, dim=[0, 1]).item():.4f}")
print(f"∀ over the 8 diagonal: {forall(paired, dim=[0]).item():.4f}")

The sizes are checked: pairing 8 individuals with 6 is an error. We try it on a copy, so that `interp` stays as it is. `∃ (x, y)` works the same way.

In [ ]:
uneven = interp.copy()
uneven["y"] = torch.randn(6, 2, device=device)
try:
    uneven("∀ (x, y): Similar(x, y)")
except ValueError as e:
    print("ValueError:", e)

## 4 — Guarded quantification

`∀ x | G(x): φ(x)` aggregates φ only over the individuals selected by the guard `G`. Below, `∀ x | IsLarge(x): IsDog(x)` reads "every large animal is a dog". The guard's truth degrees are **weights** in the mean:

- an individual with `G(x) = 0` is left out, one with `G(x) = 1` counts fully, and values in between count partially (a soft guard);
- with a crisp guard (0 or 1), the result is the same as quantifying over the selected individuals only (here, the large animals), as shown below;
- if the guard selects nobody, `∀` gives 1 and `∃` gives 0.

This is useful for **partially labelled** datasets: unlabelled individuals don't count. Ground the guard like any other predicate, as a function of its input. Don't return a tensor computed in advance from `interp["x"]`: it would be wrong once `x` is grounded with other data, and it would fail on `IsLarge(y)`, since `y` has a different number of animals.

In [ ]:
sig.predicate("IsLarge", ["Animal"])
interp["IsLarge"] = lambda a: (a[:, :1] > 0).float()   # 1 if the size is positive, computed from the input

print("∀ x | IsLarge(x): IsDog(x)   :", round(interp("∀ x | IsLarge(x): IsDog(x)").value.item(), 4))

# The same as ∀ over the large animals only
large = interp.copy()
large["x"] = interp["x"][interp["x"][:, 0] > 0]
print("∀ x: IsDog(x), large x only  :", round(large("∀ x: IsDog(x)").value.item(), 4),
      f"  ({len(large['x'])} of {len(interp['x'])} animals)")

## 5 — Changing the operators

`interp.logic` is a mutable `Logic`. Operators are read and written like a dictionary, with the logic symbols as keys: `∧` / `&` / `and`, `∨` / `or`, `¬` / `not`, `→` / `->`, `↔` / `<->`, `∀` / `forall`, `∃` / `exists`, and `SatAgg`.

- Changing an operator on a **copy** leaves the original unchanged.
- An operator you build yourself is **stable by default**, like the defaults; pass `stable=False` for the exact operator.
- `interp.logic = Logic()` resets every operator to its default.

In [ ]:
from sltn.interpretation.operators import aggregator, conjunction, implication

# A copy with a stricter ∀ (p = 4): the original is unchanged
strict = interp.copy()
strict.logic["∀"] = aggregator.AggregPMeanError(p=4)
print("∀ in interp :", interp.logic["∀"])
print("∀ in strict :", strict.logic["∀"])

phi = sig.parse("forall x: IsDog(x)")
print(f"forall x: IsDog(x)  ->  interp {interp(phi).value.item():.4f}   strict {strict(phi).value.item():.4f}")

# In place: Gödel conjunction and Goguen implication (the residuum of the product)
interp.logic["&"] = conjunction.AndMin()
interp.logic["->"] = implication.ImpliesGoguen()
print("\n∧ :", interp.logic["&"])
print("→ :", interp.logic["->"])

# Back to the defaults
interp.logic = Logic()
print("\n→ after the reset:", interp.logic["->"])

---

**Next:** [Tutorial 3 — Fuzzy Operators & Gradients](3-FuzzyOperators&Gradients.ipynb) looks at *why* the choice of connective and aggregation operator matters for learning.